# 05 — Comprehensive Cross-Sector Analysis

6-section analysis of cross-sector transient correlations:

1. **Pair Discovery Landscape** — Frequency tiers, cross-sector counts, sector heatmap
2. **Relationship Validation** — 5-test scored framework
3. **Recurring Pairs** — Most consistent co-clustering pairs
4. **Trading Viability** — Baseline vs enhanced backtests
5. **Semiconductor Baseline Comparison** — vs 54% profitable, Sharpe 3.55
6. **Key Findings & Limitations**

**Semiconductor baseline:** 54% profitable, top Sharpe 3.55

In [ ]:
import sys, os

project_root = os.path.abspath(os.path.join('..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from screener.analysis import (
    build_pair_registry, run_analysis,
    pair_type_summary, sector_pair_breakdown, generate_report,
    validate_pair_relationship,
)
from screener.enhanced_backtest import (
    run_enhanced_analysis, enhanced_backtest_pair, compute_annualized_sharpe,
)
from trading.trading import (
    walk_forward_backtest, get_daily_prices, compute_noise_adjusted_frequency,
)
from screener.universe import load_cached_universe

%matplotlib inline
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
# Load artifacts
data_dir = os.path.join('..', 'data', 'combined')
_, _, sector_map = load_cached_universe('combined')

with open(os.path.join(data_dir, 'ts_df.pkl'), 'rb') as f:
    ts_df = pickle.load(f)
with open(os.path.join(data_dir, 'cluster_history.pkl'), 'rb') as f:
    cluster_history = pickle.load(f)
with open(os.path.join(data_dir, 'pair_co_cluster_freq.pkl'), 'rb') as f:
    pair_freq = pickle.load(f)

# Load permutation results from notebook 04 if available
perm_path = os.path.join(data_dir, 'permutation_results.pkl')
perm_results = None
if os.path.exists(perm_path):
    with open(perm_path, 'rb') as f:
        perm_results = pickle.load(f)
    print(f"Permutation results loaded: {len(perm_results.get('pair_zscores', {}))} pairs tested")

print(f"Sector map: {len(sector_map)} tickers")
print(f"Total co-clustering pairs: {len(pair_freq):,}")

---
## Section 1: Pair Discovery Landscape

How many pairs survive at different noise-adjusted frequency thresholds?
Where do cross-sector pairs appear?

In [ ]:
# Build registries at multiple thresholds
thresholds = [0.05, 0.08, 0.10, 0.15]
tier_stats = []

for thresh in thresholds:
    reg = build_pair_registry(
        cluster_history, pair_freq, sector_map=sector_map,
        noise_adj_freq_threshold=thresh,
    )
    n_total = len(reg)
    n_intra = (reg['pair_type'] == 'intra-sector').sum() if 'pair_type' in reg.columns else 0
    n_cross = (reg['pair_type'] == 'cross-sector').sum() if 'pair_type' in reg.columns else 0
    tier_stats.append({
        'threshold': thresh,
        'total_pairs': n_total,
        'intra_sector': int(n_intra),
        'cross_sector': int(n_cross),
    })

df_tiers = pd.DataFrame(tier_stats)
print("Frequency Tier Analysis")
print("=" * 60)
display(df_tiers)

In [ ]:
# Top 15 cross-sector pairs (at lowest threshold to see them)
reg_low = build_pair_registry(
    cluster_history, pair_freq, sector_map=sector_map,
    noise_adj_freq_threshold=0.05,
)
cross = reg_low[reg_low['pair_type'] == 'cross-sector'].copy()

print(f"\nTop 15 Cross-Sector Pairs (threshold=0.05)")
print("=" * 60)
if len(cross) > 0:
    display(cross.head(15)[['Pair', 'noise_adj_freq', 'sector_1', 'sector_2']])
else:
    print("No cross-sector pairs found.")

In [ ]:
# Sector combination heatmap
reg_08 = build_pair_registry(
    cluster_history, pair_freq, sector_map=sector_map,
    noise_adj_freq_threshold=0.08,
)

if 'sector_1' in reg_08.columns and len(reg_08) > 0:
    # Build symmetric count matrix
    sectors = sorted(set(reg_08['sector_1'].unique()) | set(reg_08['sector_2'].unique()))
    heatmap_data = pd.DataFrame(0, index=sectors, columns=sectors)
    for _, r in reg_08.iterrows():
        s1, s2 = r['sector_1'], r['sector_2']
        heatmap_data.loc[s1, s2] += 1
        if s1 != s2:
            heatmap_data.loc[s2, s1] += 1

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(heatmap_data, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
    ax.set_title(f'Sector Pair Counts (noise-adj freq >= 0.08, {len(reg_08)} pairs)')
    plt.tight_layout()
    plt.show()

    print(f"\nPair type breakdown at 0.08 threshold:")
    print(reg_08['pair_type'].value_counts().to_string())

---
## Section 2: Relationship Validation

5-test scored framework: ADF | Half-Life | Hurst | Variance Ratio | Rolling Correlation

- **Strong (4-5/5):** High-confidence pair
- **Moderate (3/5):** Likely valid relationship
- **Weak (2/5):** Marginal evidence
- **Fail (<2/5):** Insufficient evidence

In [ ]:
# Run full analysis with 5-test validation at 0.08 threshold
registry = reg_08.copy()
print(f"Running 5-test validation on {len(registry)} pairs...")
results = run_analysis(registry, ts_df)

print(f"\nDone. {len(results)} pairs tested.")

In [ ]:
# Score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of scores
score_counts = results['score'].value_counts().sort_index()
colors = {0: '#d32f2f', 1: '#f57c00', 2: '#fbc02d', 3: '#7cb342', 4: '#388e3c', 5: '#1b5e20'}
bar_colors = [colors.get(s, '#666') for s in score_counts.index]
axes[0].bar(score_counts.index, score_counts.values, color=bar_colors)
axes[0].set_xlabel('Validation Score (out of 5)')
axes[0].set_ylabel('Number of Pairs')
axes[0].set_title('Score Distribution')
axes[0].set_xticks(range(6))

# Classification breakdown
cls_order = ['strong', 'moderate', 'weak', 'fail']
cls_counts = results['classification'].value_counts().reindex(cls_order, fill_value=0)
cls_colors = ['#388e3c', '#7cb342', '#fbc02d', '#d32f2f']
axes[1].bar(cls_counts.index, cls_counts.values, color=cls_colors)
axes[1].set_ylabel('Number of Pairs')
axes[1].set_title('Classification Breakdown')

plt.tight_layout()
plt.show()

print("Classification summary:")
for cls in cls_order:
    n = int((results['classification'] == cls).sum())
    print(f"  {cls.capitalize():10s}: {n}")

In [ ]:
# Test-by-test pass rates
valid = results[results['n_daily_obs'] > 0]
tests = [
    ('adf_passed', 'ADF (p<0.10)'),
    ('hl_passed', 'Half-life (5-60d)'),
    ('hurst_passed', 'Hurst (<0.5)'),
    ('vr_passed', 'Variance Ratio'),
    ('rc_passed', 'Rolling Corr Stability'),
]

print("Test-by-test pass rates (diagnostic):")
print("=" * 50)
test_rates = []
for col, label in tests:
    if col in valid.columns:
        rate = valid[col].mean()
        n_pass = int(valid[col].sum())
        print(f"  {label:25s}: {rate:5.1%}  ({n_pass}/{len(valid)})")
        test_rates.append({'test': label, 'pass_rate': rate, 'n_pass': n_pass})

# Visualize
if test_rates:
    df_rates = pd.DataFrame(test_rates)
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(df_rates['test'], df_rates['pass_rate'], color='steelblue')
    ax.set_xlim(0, 1)
    ax.set_xlabel('Pass Rate')
    ax.set_title('Individual Test Pass Rates')
    for bar, rate in zip(bars, df_rates['pass_rate']):
        ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                f'{rate:.0%}', va='center')
    plt.tight_layout()
    plt.show()

In [ ]:
# Breakdown by pair type and sector
if 'pair_type' in results.columns:
    print("Score by pair type:")
    print("=" * 50)
    ptype = pair_type_summary(results)
    display(ptype)

    print("\nSector pair breakdown:")
    spb = sector_pair_breakdown(results)
    display(spb)

In [ ]:
# Table of strong + moderate pairs with all test details
tradeable = results[results['score'] >= 3].copy()
print(f"\nStrong + Moderate pairs: {len(tradeable)}")
print("=" * 80)

detail_cols = ['pair', 'score', 'classification', 'noise_adj_freq',
               'adf_pval', 'half_life_days', 'hurst', 'variance_ratio', 'rolling_corr_stability']
if 'pair_type' in tradeable.columns:
    detail_cols = ['pair_type'] + detail_cols

if len(tradeable) > 0:
    display(tradeable.sort_values(['score', 'noise_adj_freq'], ascending=[False, False])[detail_cols])
else:
    print("No pairs scored >= 3.")

---
## Section 3: Recurring Pairs

Which pairs co-cluster most frequently? Are recurring pairs also well-validated?

In [ ]:
# Pairs with highest co-cluster frequency
print("Top 20 pairs by noise-adjusted frequency:")
print("=" * 60)

top_freq = results.nlargest(20, 'noise_adj_freq')
freq_cols = ['pair', 'noise_adj_freq', 'raw_freq', 'score', 'classification']
if 'pair_type' in top_freq.columns:
    freq_cols = ['pair_type'] + freq_cols
display(top_freq[freq_cols])

In [ ]:
# Formation counts and durations from cached artifacts
dur_path = os.path.join(data_dir, 'df_durations.pkl')
form_path = os.path.join(data_dir, 'df_formations.pkl')

if os.path.exists(dur_path) and os.path.exists(form_path):
    with open(dur_path, 'rb') as f:
        df_durations = pickle.load(f)
    with open(form_path, 'rb') as f:
        df_formations = pickle.load(f)

    # Build pair key from formations
    if 'Pair' in df_formations.columns:
        form_counts = df_formations['Pair'].value_counts().head(20)
        print(f"\nTop 20 pairs by formation count:")
        print(form_counts.to_string())

    if 'Pair' in df_durations.columns and 'Duration_Hours' in df_durations.columns:
        avg_dur = df_durations.groupby('Pair')['Duration_Hours'].mean().sort_values(ascending=False).head(20)
        print(f"\nTop 20 pairs by avg formation duration (hours):")
        print(avg_dur.to_string())
else:
    print("Formation/duration artifacts not found — skipping.")

In [ ]:
# Statistically significant pairs from permutation test
if perm_results is not None:
    pair_zscores = perm_results.get('pair_zscores', {})
    sig_pairs = {p for p, z in pair_zscores.items() if z > 1.96}
    print(f"Statistically significant pairs (permutation Z > 1.96): {len(sig_pairs)}")

    # Overlap: significant AND validated (score >= 3)
    validated_pairs = set()
    for _, r in tradeable.iterrows():
        validated_pairs.add((r['ticker_a'], r['ticker_b']))
        validated_pairs.add(r['pair'])

    overlap = []
    for pair_key in sig_pairs:
        if isinstance(pair_key, tuple):
            pair_str = f"{pair_key[0]}-{pair_key[1]}"
        else:
            pair_str = pair_key
        if pair_str in validated_pairs or pair_key in validated_pairs:
            overlap.append(pair_str)

    print(f"Overlap (significant + score >= 3): {len(overlap)}")
    if overlap:
        print("  Pairs:", sorted(overlap))
else:
    print("No permutation results available.")

---
## Section 4: Trading Viability

Compare baseline (static z=2.0, OLS, no costs) vs enhanced (optimized z, Kalman, 10bps costs).

In [ ]:
# Run enhanced backtests on tradeable pairs (score >= 3)
tradeable_reg = registry[registry['Pair'].isin(tradeable['pair'])].copy()

if len(tradeable_reg) > 0:
    print(f"Running enhanced backtests on {len(tradeable_reg)} tradeable pairs...")
    enhanced_results = run_enhanced_analysis(tradeable_reg, ts_df, cost_per_trade=0.001)
    print(f"Done. {len(enhanced_results)} pairs backtested.")
else:
    enhanced_results = pd.DataFrame()
    print("No tradeable pairs to backtest.")

In [ ]:
# Side-by-side comparison table
if len(enhanced_results) > 0:
    compare_cols = ['pair']
    if 'pair_type' in enhanced_results.columns:
        compare_cols.append('pair_type')
    compare_cols += [
        'baseline_n_trades', 'baseline_pnl', 'baseline_sharpe', 'baseline_ann_sharpe',
        'opt_entry_z', 'opt_lookback',
        'enhanced_n_trades', 'enhanced_pnl', 'enhanced_sharpe', 'enhanced_ann_sharpe',
        'kalman_n_trades', 'kalman_pnl', 'kalman_sharpe', 'kalman_ann_sharpe',
    ]
    available_cols = [c for c in compare_cols if c in enhanced_results.columns]

    print("Baseline vs Enhanced vs Kalman:")
    print("=" * 80)
    display(enhanced_results[available_cols].sort_values('enhanced_pnl', ascending=False))

In [ ]:
# Strategy comparison summary
if len(enhanced_results) > 0:
    strategies = [
        ('Baseline (z=2.0, OLS, no costs)', 'baseline'),
        ('Enhanced (opt z, OLS, 10bps)', 'enhanced'),
        ('Kalman (opt z, Kalman, 10bps)', 'kalman'),
    ]

    summary_rows = []
    for label, prefix in strategies:
        pnl_col = f'{prefix}_pnl'
        trades_col = f'{prefix}_n_trades'
        wr_col = f'{prefix}_win_rate'
        sharpe_col = f'{prefix}_sharpe'
        ann_col = f'{prefix}_ann_sharpe'
        dd_col = f'{prefix}_max_dd'

        if pnl_col not in enhanced_results.columns:
            continue

        valid = enhanced_results[enhanced_results[trades_col] > 0]
        summary_rows.append({
            'Strategy': label,
            'Pairs w/ trades': len(valid),
            'Avg trades': valid[trades_col].mean() if len(valid) else 0,
            'Avg PnL': valid[pnl_col].mean() if len(valid) else np.nan,
            'Profitable %': (valid[pnl_col] > 0).mean() if len(valid) else np.nan,
            'Avg win rate': valid[wr_col].mean() if len(valid) else np.nan,
            'Avg Sharpe': valid[sharpe_col].dropna().mean() if len(valid) else np.nan,
            'Avg Ann Sharpe': valid[ann_col].dropna().mean() if len(valid) else np.nan,
        })

    df_summary = pd.DataFrame(summary_rows)
    print("Strategy Comparison Summary:")
    print("=" * 80)
    display(df_summary)

In [ ]:
# Walk-forward on top 10 enhanced pairs
if len(enhanced_results) > 0:
    top_enhanced = enhanced_results.dropna(subset=['enhanced_pnl']).nlargest(10, 'enhanced_pnl')

    wf_results = []
    for _, row in top_enhanced.iterrows():
        daily_a = get_daily_prices(row['ticker_a'], ts_df)
        daily_b = get_daily_prices(row['ticker_b'], ts_df)

        # Use optimized params if available
        ez = row.get('opt_entry_z', 2.0)
        lb = int(row.get('opt_lookback', 20)) if not np.isnan(row.get('opt_lookback', np.nan)) else 20
        xz = row.get('opt_exit_z', 0.5)
        if np.isnan(ez): ez = 2.0
        if np.isnan(xz): xz = 0.5

        wf = walk_forward_backtest(daily_a, daily_b, entry_z=ez, exit_z=xz, lookback=lb)
        if wf is not None:
            wf_entry = {
                'pair': row['pair'],
                'n_splits': wf['n_splits'],
                'avg_sharpe': wf['avg_sharpe'],
                'std_sharpe': wf['std_sharpe'],
                'avg_pnl': wf['avg_pnl'],
                'total_trades': wf['total_trades'],
            }
            if 'pair_type' in row.index:
                wf_entry['pair_type'] = row['pair_type']
            wf_results.append(wf_entry)

    df_wf = pd.DataFrame(wf_results)
    if not df_wf.empty:
        print("Walk-Forward Results (top 10 enhanced pairs):")
        print("=" * 60)
        display(df_wf.sort_values('avg_sharpe', ascending=False))
    else:
        print("No pairs qualified for walk-forward validation.")

In [ ]:
# Distribution plots
if len(enhanced_results) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # P&L distributions
    for prefix, label, color in [
        ('baseline', 'Baseline', 'steelblue'),
        ('enhanced', 'Enhanced', 'darkorange'),
        ('kalman', 'Kalman', 'forestgreen'),
    ]:
        col = f'{prefix}_pnl'
        if col in enhanced_results.columns:
            vals = enhanced_results[col].dropna()
            if len(vals) > 0:
                axes[0].hist(vals, bins=15, alpha=0.5, label=label, color=color)
    axes[0].axvline(0, color='black', linestyle='--', alpha=0.5)
    axes[0].set_title('P&L Distribution')
    axes[0].set_xlabel('Total P&L')
    axes[0].legend()

    # Trade counts
    for prefix, label, color in [
        ('baseline', 'Baseline', 'steelblue'),
        ('enhanced', 'Enhanced', 'darkorange'),
    ]:
        col = f'{prefix}_n_trades'
        if col in enhanced_results.columns:
            vals = enhanced_results[col].dropna()
            if len(vals) > 0:
                axes[1].hist(vals, bins=15, alpha=0.5, label=label, color=color)
    axes[1].set_title('Trade Count Distribution')
    axes[1].set_xlabel('Number of OOS Trades')
    axes[1].legend()

    # Sharpe comparison
    for prefix, label, color in [
        ('baseline', 'Baseline', 'steelblue'),
        ('enhanced', 'Enhanced', 'darkorange'),
        ('kalman', 'Kalman', 'forestgreen'),
    ]:
        col = f'{prefix}_sharpe'
        if col in enhanced_results.columns:
            vals = enhanced_results[col].dropna()
            if len(vals) > 0:
                axes[2].hist(vals, bins=15, alpha=0.5, label=label, color=color)
    axes[2].axvline(0, color='black', linestyle='--', alpha=0.5)
    axes[2].set_title('Sharpe Distribution')
    axes[2].set_xlabel('Sharpe Ratio')
    axes[2].legend()

    plt.tight_layout()
    plt.show()

---
## Section 5: Semiconductor Baseline Comparison

Semiconductor baseline: 54% profitable, top Sharpe 3.55

In [ ]:
semi_baseline = {'profitable_frac': 0.54, 'top_sharpe': 3.55}

print("=" * 60)
print("COMPARISON vs SEMICONDUCTOR BASELINE")
print("=" * 60)
print(f"Baseline: {semi_baseline['profitable_frac']:.0%} profitable, top Sharpe {semi_baseline['top_sharpe']:.2f}")

# Screened universe results
if len(tradeable) > 0:
    profitable = (tradeable['total_pnl'] > 0).mean()
    valid_s = tradeable['sharpe'].dropna()
    top_sharpe = valid_s.max() if len(valid_s) > 0 else np.nan
    avg_sharpe = valid_s.mean() if len(valid_s) > 0 else np.nan

    print(f"\nScreened universe (5-test validation, threshold=0.08):")
    print(f"  Total pairs tested: {len(results)}")
    print(f"  Tradeable (score>=3): {len(tradeable)}")
    print(f"  Profitable: {profitable:.0%}" if len(tradeable) > 0 else "  Profitable: N/A")
    print(f"  Top Sharpe: {top_sharpe:.2f}" if np.isfinite(top_sharpe) else "  Top Sharpe: N/A")
    print(f"  Avg Sharpe: {avg_sharpe:.2f}" if np.isfinite(avg_sharpe) else "  Avg Sharpe: N/A")

    # Intra vs cross comparison
    if 'pair_type' in tradeable.columns:
        print()
        for ptype in ['intra-sector', 'cross-sector']:
            sub = tradeable[tradeable['pair_type'] == ptype]
            if len(sub) > 0:
                p = (sub['total_pnl'] > 0).mean()
                vs = sub['sharpe'].dropna()
                ts = vs.max() if len(vs) > 0 else np.nan
                avg_s = vs.mean() if len(vs) > 0 else np.nan
                print(f"  {ptype}:")
                print(f"    Pairs: {len(sub)}, Profitable: {p:.0%}")
                if np.isfinite(ts):
                    print(f"    Top Sharpe: {ts:.2f}, Avg Sharpe: {avg_s:.2f}")

    # Enhanced comparison
    if len(enhanced_results) > 0:
        enh_valid = enhanced_results[enhanced_results['enhanced_n_trades'] > 0]
        if len(enh_valid) > 0:
            enh_profitable = (enh_valid['enhanced_pnl'] > 0).mean()
            enh_sharpes = enh_valid['enhanced_sharpe'].dropna()
            enh_top = enh_sharpes.max() if len(enh_sharpes) > 0 else np.nan
            print(f"\n  Enhanced strategy (optimized z, 10bps costs):")
            print(f"    Pairs w/ trades: {len(enh_valid)}")
            print(f"    Profitable: {enh_profitable:.0%}")
            if np.isfinite(enh_top):
                print(f"    Top Sharpe: {enh_top:.2f}")
else:
    print("\nNo tradeable pairs to compare.")

In [ ]:
# Visual comparison
if len(tradeable) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Profitability comparison
    labels = ['Semiconductor\nBaseline']
    values_prof = [semi_baseline['profitable_frac']]
    colors_prof = ['#1f77b4']

    overall_prof = (tradeable['total_pnl'] > 0).mean()
    labels.append('Cross-Sector\n(all)')
    values_prof.append(overall_prof)
    colors_prof.append('#ff7f0e')

    if 'pair_type' in tradeable.columns:
        for ptype, color in [('intra-sector', '#2ca02c'), ('cross-sector', '#d62728')]:
            sub = tradeable[tradeable['pair_type'] == ptype]
            if len(sub) > 0:
                labels.append(f'{ptype}')
                values_prof.append((sub['total_pnl'] > 0).mean())
                colors_prof.append(color)

    axes[0].bar(labels, values_prof, color=colors_prof)
    axes[0].set_ylabel('Profitable Fraction')
    axes[0].set_title('Profitability Comparison')
    axes[0].set_ylim(0, 1)
    axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5)

    # Sharpe comparison
    labels_s = ['Semi Baseline']
    values_s = [semi_baseline['top_sharpe']]
    colors_s = ['#1f77b4']

    valid_s = tradeable['sharpe'].dropna()
    if len(valid_s) > 0:
        labels_s.append('Cross-Sector\nTop')
        values_s.append(valid_s.max())
        colors_s.append('#ff7f0e')

    if len(enhanced_results) > 0:
        enh_s = enhanced_results['enhanced_sharpe'].dropna()
        if len(enh_s) > 0:
            labels_s.append('Enhanced\nTop')
            values_s.append(enh_s.max())
            colors_s.append('#2ca02c')

    axes[1].bar(labels_s, values_s, color=colors_s)
    axes[1].set_ylabel('Sharpe Ratio')
    axes[1].set_title('Top Sharpe Comparison')

    plt.tight_layout()
    plt.show()

---
## Section 6: Key Findings & Limitations

In [ ]:
# Generate full report
report = generate_report(results)
print(report)

In [ ]:
# Summary table
summary_data = {
    'Metric': [
        'Total pairs at 0.08 threshold',
        'Intra-sector pairs',
        'Cross-sector pairs',
        'Strong (score 4-5)',
        'Moderate (score 3)',
        'Weak (score 2)',
        'Fail (score <2)',
        'ADF pass rate',
        'Half-life pass rate',
        'Hurst pass rate',
        'Variance ratio pass rate',
        'Rolling corr pass rate',
    ],
    'Value': [
        len(results),
        int((results.get('pair_type', pd.Series()) == 'intra-sector').sum()),
        int((results.get('pair_type', pd.Series()) == 'cross-sector').sum()),
        int((results['classification'] == 'strong').sum()),
        int((results['classification'] == 'moderate').sum()),
        int((results['classification'] == 'weak').sum()),
        int((results['classification'] == 'fail').sum()),
        f"{valid['adf_passed'].mean():.0%}" if 'adf_passed' in valid.columns else 'N/A',
        f"{valid['hl_passed'].mean():.0%}" if 'hl_passed' in valid.columns else 'N/A',
        f"{valid['hurst_passed'].mean():.0%}" if 'hurst_passed' in valid.columns else 'N/A',
        f"{valid['vr_passed'].mean():.0%}" if 'vr_passed' in valid.columns else 'N/A',
        f"{valid['rc_passed'].mean():.0%}" if 'rc_passed' in valid.columns else 'N/A',
    ],
}
df_final_summary = pd.DataFrame(summary_data)
display(df_final_summary)

print("\nLimitations:")
print("- Data covers ~10 months of hourly data; longer periods needed for robustness")
print("- OOS window is ~68 days — short for reliable Sharpe estimation")
print("- Transaction costs are simplified (flat 10bps); real costs vary by asset")
print("- Kalman filter requires tuning of transition covariance (currently fixed at 0.01)")
print("- Cross-sector pairs have lower frequency by construction — fewer co-clustering events")

In [ ]:
# Save all results
with open(os.path.join(data_dir, 'analysis_results.pkl'), 'wb') as f:
    pickle.dump(results, f)

with open(os.path.join(data_dir, 'report.txt'), 'w') as f:
    f.write(report)

if len(enhanced_results) > 0:
    with open(os.path.join(data_dir, 'enhanced_results.pkl'), 'wb') as f:
        pickle.dump(enhanced_results, f)

if 'df_wf' in dir() and not df_wf.empty:
    with open(os.path.join(data_dir, 'walk_forward_results.pkl'), 'wb') as f:
        pickle.dump(df_wf, f)

print("All results saved.")